## Imports & Config

In [116]:
import os
import time
import fitz  # PyMuPDF
import pandas as pd
import numpy as np
import matplotlib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# ----------- CONFIG -----------
BASE_PATH = r"/mnt/d/Arunachalam/Projects/CBCID/Data/FIR"   # <<< CHANGE THIS
ZONES = ["CZ", "NZ", "SZ", "WZ"]

OUT_DIR = "../data_exploration_graphs"
os.makedirs(OUT_DIR, exist_ok=True)


## Helper Functions

In [117]:
def extract_year_from_filename(fname):
    """Simple heuristic: detect a 4-digit year inside the file name."""
    tokens = fname.replace("-", "_").split("_")
    for t in tokens:
        if t.isdigit() and len(t) == 4 and 1900 <= int(t) <= 2100:
            return int(t)
    return None

def safe_get_page_count(pdf_path):
    """Return page count; skip unreadable PDFs."""
    try:
        doc = fitz.open(pdf_path)
        n = doc.page_count
        doc.close()
        return n
    except Exception as e:
        print(f"⚠ Error reading pages for {pdf_path}: {e}")
        return None


## Scan All Zones & Build DataFrame

In [118]:

records = []

print("\n Scanning zone folders...\n")

for zone in ZONES:
    zone_folder = os.path.join(BASE_PATH, zone)
    if not os.path.exists(zone_folder):
        print(f"⚠ Missing zone folder: {zone_folder}")
        continue

    pdf_files = [f for f in os.listdir(zone_folder) if f.lower().endswith(".pdf")]
    print(f" Zone {zone}: {len(pdf_files)} PDFs found")

    for fname in tqdm(pdf_files, desc=f"Processing {zone}", unit="file"):
        full_path = os.path.join(zone_folder, fname)

        # Metadata
        size_kb = os.path.getsize(full_path) / 1024
        mtime = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(os.path.getmtime(full_path)))

        # Page count
        page_count = safe_get_page_count(full_path)

        records.append({
            "zone": zone,
            "filename": fname,
            "size_kb": size_kb,
            "page_count": page_count
        })

df = pd.DataFrame(records)
df.head()



 Scanning zone folders...

 Zone CZ: 5 PDFs found


Processing CZ:   0%|          | 0/5 [00:00<?, ?file/s]

Processing CZ: 100%|██████████| 5/5 [00:00<00:00, 19.80file/s]


 Zone NZ: 7 PDFs found


Processing NZ: 100%|██████████| 7/7 [00:00<00:00, 11.55file/s]


 Zone SZ: 2 PDFs found


Processing SZ: 100%|██████████| 2/2 [00:00<00:00, 14.74file/s]


 Zone WZ: 42 PDFs found


Processing WZ: 100%|██████████| 42/42 [00:03<00:00, 12.62file/s]


,zone,filename,size_kb,page_count
0,CZ,Ariyalur CBCID Cr.No.01-23 Suspetious death.pdf,3583.409180,6
1,CZ,Karur Velayutham palayam PS.Cr.No.339 of 2011 ...,1128.146484,2
2,CZ,Nagapatinam Cr.No.01 of 2015 Double murder cas...,1530.691406,3
3,CZ,Thiruvarur CBCID Cr.No.01 of 2015 Women missin...,4176.633789,3
4,CZ,Trichy CBCID Cr.No.02 of 2022 Suspetious drawi...,1939.182617,4


## Basic Stats

In [119]:
print("Total FIR PDFs:", len(df))
print("\nCount per zone:")
display(df["zone"].value_counts())

print("\nMissing page counts:")
display(df[df["page_count"].isnull()][["zone", "filename"]])


Total FIR PDFs: 56

Count per zone:


zone
WZ    42
NZ     7
CZ     5
SZ     2
Name: count, dtype: int64


Missing page counts:


,zone,filename


## FIR Count per Zone (PNG)

In [120]:
plt.figure(figsize=(7,5))

ax = sns.countplot(data=df, x="zone", order=ZONES)

# Add integer labels above bars
for bar in ax.patches:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        str(int(height)),   #  force integer
        ha='center', va='bottom',
        fontsize=11, fontweight='bold'
    )

plt.title("FIR Count per Zone")
plt.xlabel("Zone")
plt.ylabel("Number of FIRs")
plt.tight_layout()

path = f"{OUT_DIR}/fir_count_per_zone.png"
plt.savefig(path, dpi=150)
plt.close()

print("Saved:", path)


Saved: ../data_exploration_graphs/fir_count_per_zone.png


## Page Count Distribution (Histogram)

In [121]:
df_pages = df[df["page_count"].notnull()].copy()

plt.figure(figsize=(8,5))
ax = sns.countplot(data=df_pages, x="page_count")
# Add integer labels above bars
for bar in ax.patches:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        str(int(height)),   #  force integer
        ha='center', va='bottom',
        fontsize=11, fontweight='bold'
    )
plt.title("Page Count Distribution")
plt.xlabel("Page Count")
plt.ylabel("Number of FIRs")
plt.tight_layout()

path = f"{OUT_DIR}/page_count_histogram.png"
plt.savefig(path, dpi=150)
plt.close()

print("Saved:", path)


Saved: ../data_exploration_graphs/page_count_histogram.png


## Page Count Violin Plot per Zone

In [122]:
plt.figure(figsize=(9,6))
sns.violinplot(data=df_pages, x="zone", y="page_count", order=ZONES)

plt.title("Page Count Distribution per Zone")
plt.xlabel("Zone")
plt.ylabel("Page Count")

plt.tight_layout()

path = f"{OUT_DIR}/page_distribution_per_zone.png"
plt.savefig(path, dpi=150, bbox_inches='tight')  # FIX: add bbox_inches
plt.close()

print("Saved:", path)

Saved: ../data_exploration_graphs/page_distribution_per_zone.png
